# Lesson 4.4: create_agent + middleware = the same loop, line by line

**Goal:** rebuild Nightwatch v0 with LangChain's `create_agent` and see that every line of your hand-written loop moved
into one named piece: the model, the tool node, a call limit, or a middleware.

**What you will be able to do**
- Call `create_agent` with v0's model settings, prompt, 17 tools and a middleware list.
- Write the three kinds of hooks yourself: `wrap_model_call`, `after_model`, `wrap_tool_call`.
- Predict hook order: wrap hooks nest (first listed = outermost), `after_model` hooks run in reverse list order.
- Compare v0 and create_agent on the same incident, and say what one run each can and cannot prove.

In [1]:
# Setup: find the course repo's nightwatch/ folder (relative to this notebook) and load .env if present.
import os, sys
from pathlib import Path

for up in ("../../nightwatch", "../../../nightwatch", "../nightwatch", "nightwatch"):
    if Path(up, "brightcart").is_dir():
        NW = Path(up)
        break
else:
    raise FileNotFoundError("Open this notebook from <lesson>/notebook inside the course repo: it needs the nightwatch/ folder.")
sys.path.insert(0, str(NW.resolve()))

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))      # walks up from here and loads the repo's .env
except ImportError:
    pass
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
print("nightwatch folder:", NW.as_posix(), "| GROQ_API_KEY:", "found" if HAS_KEY else "not set")
if not HAS_KEY:
    print("No GROQ_API_KEY: get the free key as shown in lesson 0.2, put GROQ_API_KEY=... in the repo's .env,")
    print("then restart. Cells that need it: only section 6c (one real run of expired_secret-1); it replays from runs/lc_cache, so a rerun costs nothing. They replay from the cache when it has the answer.")

nightwatch folder: ../../../nightwatch | GROQ_API_KEY: found


## 1. Recall: after the tools node (lesson 4.3)

Last lesson the loop became a `model` node, a `tools` node and one conditional edge. Which node runs right after `tools`?

In [2]:
edges_43 = {"model": "tools if the reply has tool calls, else END", "tools": "model"}
print("after tools ->", edges_43["tools"], "(a plain edge: no decision)")

after tools -> model (a plain edge: no decision)


## 2. Key words

| Word | Meaning here |
|---|---|
| **create_agent** | one LangChain function that builds the model + tools loop for you |
| **middleware** | code you plug into that loop without editing it |
| **hook** | where middleware runs: `before_model`, `after_model`, or wrapped around a call (`wrap_model_call`, `wrap_tool_call`) |
| **return_direct** | a tool flag: calling that tool ends the run |

## 3. The idea, and a hand trace

**v0's lines, and the piece that replaced each one** (from `nw_agent.py`):

| v0 `run()` line | create_agent piece |
|---|---|
| `msgs = [system, PAGER]` | `system_prompt=SYSTEM`, `{"messages": [PAGER]}` |
| `for step in range(14)` | `ModelCallLimitMiddleware(run_limit=14)` |
| `out = chat(body)` | `ChatGroq` (gpt-oss-20b, temperature 1.0, reasoning medium, 2048 tokens) |
| chat()'s record/replay cache | `@wrap_model_call record_replay` |
| `if "error" in out: nudge, continue` | `@wrap_model_call retry_malformed` |
| `if not calls: "Use the tools..."` | `@after_model nudge_to_tools` (jump back to the model) |
| `for c in calls: run_tool(...)` | the 17 tools as `StructuredTool`s (the tool node runs them) |
| `trace.append(...)` | `@wrap_tool_call log_tool` |
| `if name == "finish": break` | `finish` tool with `return_direct=True` |

Nine lines, nine pieces: nothing new, it just moved.

**One run walked by hand: `expired_secret-1`** (pager: *checkout error rate above 5%*), counting model calls:

| model call | tool it asks for | what comes back |
|---|---|---|
| 1 | check_health | checkout and payments unhealthy |
| 2 | recent_deploys | one deploy, inventory v2.4.9 (not related) |
| 3 | read_logs(checkout) | "upstream payments returned 502" |
| 4 | read_logs(payments) | "401 Unauthorized: key expired" |
| 5 | rotate_secret(payments) | done |
| 6 | check_health | healthy |
| 7 | finish(expired_secret, payments) | return_direct ends the run |

Seven model calls, seven tool calls, one write, no retries: pass, clean, right service.

## 4. Write it yourself: the hooks, on a fake model

To see the mechanics without any API, we give `create_agent` a **scripted fake model** that replies with a fixed list
of messages, and three tiny tools. The hooks below are the same shape as Nightwatch's.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, after_model, wrap_model_call, wrap_tool_call
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

class FakeModel(GenericFakeChatModel):
    """Replies with the scripted messages, one per model call. bind_tools is a no-op."""
    def bind_tools(self, tools, **kw):
        return self

def call(name, i, **args):                       # a scripted tool-call reply
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": f"c{i}"}])

def fake(*replies):
    return FakeModel(messages=iter(replies))

WROTE = []                                       # side effects of the write tool

@tool
def check_health() -> str:
    """Check the shop's health."""
    return "unhealthy: checkout, payments" if not WROTE else "healthy"

@tool
def rotate_secret(service: str) -> str:
    """Rotate a service's secret (a write)."""
    WROTE.append(("rotate_secret", service))
    return f"rotate_secret({service}) done."

@tool(return_direct=True)
def finish(cause: str, root_cause_service: str) -> str:
    """Report the root cause and end the run."""
    return "ok"

TOOLS = [check_health, rotate_secret, finish]
print([t.name for t in TOOLS], "| finish return_direct:", finish.return_direct)

['check_health', 'rotate_secret', 'finish'] | finish return_direct: True


Now the hooks. `box` collects the trace, like v0's `trace` list.

In [4]:
box = {"trace": []}

@after_model(can_jump_to=["model"])
def nudge_to_tools(state, runtime):
    """Text with no tool call: add v0's nudge and jump back to the model."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and not last.tool_calls:
        box["trace"].append({"tool": None, "text": last.content[:60]})
        return {"messages": [HumanMessage("Use the tools. End with finish().")], "jump_to": "model"}
    return None                                  # None changes nothing: the normal edge is followed

@wrap_tool_call
def log_tool(request, handler):
    """Run the tool, then append name, args and the start of the result."""
    out = handler(request)
    box["trace"].append({"tool": request.tool_call["name"], "args": request.tool_call["args"],
                         "result": str(out.content)[:60]})
    return out

def mini_agent(model, middleware):
    return create_agent(model, TOOLS, system_prompt="You are Nightwatch.", middleware=middleware)

A scripted run: one text-only reply in the middle (the nudge fires), then the fix and finish.

In [5]:
WROTE.clear(); box["trace"].clear()
model = fake(call("check_health", 1),
             AIMessage("I think it is the payments key."),          # text only: nudge
             call("rotate_secret", 2, service="payments"),
             call("check_health", 3),
             call("finish", 4, cause="expired_secret", root_cause_service="payments"))
agent = mini_agent(model, [nudge_to_tools, ModelCallLimitMiddleware(run_limit=14, exit_behavior="end"), log_tool])
out = agent.invoke({"messages": [HumanMessage("PAGER: checkout error rate above 5%")]}, config={"recursion_limit": 100})
for i, t in enumerate(box["trace"]):
    print(i, t)
print("model calls:", sum(isinstance(m, AIMessage) for m in out["messages"]), "| writes:", WROTE)

0 {'tool': 'check_health', 'args': {}, 'result': 'unhealthy: checkout, payments'}
1 {'tool': None, 'text': 'I think it is the payments key.'}
2 {'tool': 'rotate_secret', 'args': {'service': 'payments'}, 'result': 'rotate_secret(payments) done.'}
3 {'tool': 'check_health', 'args': {}, 'result': 'healthy'}
4 {'tool': 'finish', 'args': {'cause': 'expired_secret', 'root_cause_service': 'payments'}, 'result': 'ok'}
model calls: 5 | writes: [('rotate_secret', 'payments')]


### A wrap hook: `retry_malformed`

A `wrap_model_call` function gets the request and a `handler` that makes the real call. Nightwatch's version catches a
malformed tool call, adds v0's nudge and tries again, up to three times. Here a fake model fails once, then answers.
One difference from v0: v0 counts that retry as a step, here it stays inside one model call.

In [6]:
class Malformed(Exception):
    pass

errors = []

@wrap_model_call
def retry_malformed(request, handler):
    for attempt in range(3):
        try:
            return handler(request)
        except Malformed as e:
            errors.append(str(e))
            request = request.override(messages=request.messages + [
                HumanMessage("Your last tool call was malformed. Call one tool with valid JSON.")])
    return handler(request)

class FlakyModel(FakeModel):
    fails: int = 1
    def _generate(self, messages, *a, **kw):
        if self.fails:
            self.fails -= 1
            raise Malformed("tool_use_failed: invalid JSON")
        return super()._generate(messages, *a, **kw)

WROTE.clear(); box["trace"].clear()
agent = mini_agent(FlakyModel(messages=iter([call("finish", 1, cause="false_alarm", root_cause_service="web")])),
                   [retry_malformed, log_tool])
out = agent.invoke({"messages": [HumanMessage("PAGER: web probe failed")]})
print("errors recorded:", errors)
print("trace:", [t["tool"] for t in box["trace"]], "| AI messages:", sum(isinstance(m, AIMessage) for m in out["messages"]))

errors recorded: ['tool_use_failed: invalid JSON']
trace: ['finish'] | AI messages: 1


## 5. The graph create_agent drew
Print the Mermaid edges (the class and style lines are left out).

In [7]:
def edges(agent):
    return [l.strip().replace("\\2e", ".") for l in agent.get_graph().draw_mermaid().splitlines() if "-->" in l or "-.->" in l]

g = mini_agent(fake(), [nudge_to_tools, ModelCallLimitMiddleware(run_limit=14, exit_behavior="end"), log_tool])
for e in edges(g):
    print(e)

ModelCallLimitMiddleware.after_model --> nudge_to_tools.after_model;
ModelCallLimitMiddleware.before_model -.-> __end__;
ModelCallLimitMiddleware.before_model -.-> model;
__start__ --> ModelCallLimitMiddleware.before_model;
model --> ModelCallLimitMiddleware.after_model;
nudge_to_tools.after_model -.-> ModelCallLimitMiddleware.before_model;
nudge_to_tools.after_model -.-> __end__;
nudge_to_tools.after_model -.-> tools;
tools -.-> ModelCallLimitMiddleware.before_model;
tools -.-> __end__;


`model --> ModelCallLimitMiddleware.after_model`, then `--> nudge_to_tools.after_model`: the limit's after_model runs
**first**, although `nudge_to_tools` is listed first. After-model hooks run in reverse list order. That is the order we
want: the limit counts a text-only reply before the nudge jumps back, as v0 counts it as a step.

## 6. The library version: the real Nightwatch rebuild

### 6a. Versions and the middleware stack
`nw_agent.py` (in this lesson's `examples/`) has the real thing: `ChatGroq`, v0's 17 tools, and five middleware.

In [8]:
sys.path.insert(0, str(Path("../examples").resolve()))
from importlib.metadata import version
import nw_agent
from nw_agent import agent_v0
from langchain.agents.middleware import AgentMiddleware

print(f"langchain {version('langchain')} | langgraph {version('langgraph')} | langchain-groq {version('langchain-groq')}")
print("tools:", len(agent_v0.TOOLS))
for m in nw_agent.make_middleware({"final": None, "trace": [], "errors": []}):
    hooks = [h for h in ("before_model", "after_model", "wrap_model_call", "wrap_tool_call")
             if getattr(type(m), h, None) is not getattr(AgentMiddleware, h, None)]
    print(f"  {m.name:28} hooks: {hooks}")

langchain 1.5.0 | langgraph 1.2.22 | langchain-groq 1.2.0
tools: 17
  nudge_to_tools               hooks: ['after_model']
  ModelCallLimitMiddleware     hooks: ['before_model', 'after_model']
  retry_malformed              hooks: ['wrap_model_call']
  record_replay                hooks: ['wrap_model_call']
  log_tool                     hooks: ['wrap_tool_call']


### 6b. The graph it builds (no model call: building and drawing is free)

In [9]:
from brightcart.incidents import generate
inc = next(x for x in generate() if x.id == "expired_secret-1")
b = {"final": None, "trace": [], "errors": []}
real = nw_agent.create_agent(nw_agent.make_model(), nw_agent.make_tools(nw_agent.World(nw_agent.fresh(inc)), b),
                             system_prompt=agent_v0.SYSTEM, middleware=nw_agent.make_middleware(b))
for e in edges(real):
    print(e)

ModelCallLimitMiddleware.after_model --> nudge_to_tools.after_model;
ModelCallLimitMiddleware.before_model -.-> __end__;
ModelCallLimitMiddleware.before_model -.-> model;
__start__ --> ModelCallLimitMiddleware.before_model;
model --> ModelCallLimitMiddleware.after_model;
nudge_to_tools.after_model -.-> ModelCallLimitMiddleware.before_model;
nudge_to_tools.after_model -.-> __end__;
nudge_to_tools.after_model -.-> tools;
tools -.-> ModelCallLimitMiddleware.before_model;
tools -.-> __end__;


### 6c. One run: expired_secret-1  ·  **calls the API** (replays from the cache)

`nw_agent.run` uses the `record_replay` middleware: if `runs/lc_cache` has this conversation, no request is sent.

In [10]:
r = nw_agent.run(inc, sample=0, verbose=True)
print("finish():", r["final"])
print(f"stop {r['stop_reason']} | model calls {r['model_calls']} | tool calls {r['steps']} | healthy {r['healthy']} | "
      f"writes {r['writes']} | malformed-call retries {len(r['errors'])}")
print(f"tokens prompt {r['usage']['prompt']} + completion {r['usage']['completion']}")
print("grade():", agent_v0.grade(r))

 0 check_health({"": ""}) -> {"healthy": false, "unhealthy_services": ["checkout", "payments"], "queues_over_
 1 recent_deploys({"": ""}) -> [{"t": "03:01", "service": "inventory", "version": "v2.4.9", "by": "ci"}]
 2 read_logs({"n": 20, "service": "checkout"}) -> ["03:01 INFO checkout handled request id=59367 in 76ms", "03:01 ERROR checkout u
 3 read_logs({"n": 20, "service": "payments"}) -> ["03:00 ERROR payments provider returned 401 Unauthorized: key expired", "03:00 
 4 rotate_secret({"service": "payments"}) -> "rotate_secret(payments) done."
 5 check_health({"": ""}) -> {"healthy": true, "unhealthy_services": [], "queues_over_limit": {}}
 6 finish({"cause": "expired_secret", "root_cause_service": "payments", "summary) -> ok
finish(): {'cause': 'expired_secret', 'root_cause_service': 'payments', 'summary': 'Payments provider key expired causing 401 errors; checkout forwarded 502s. Rotated secret resolved the issue.'}
stop finish | model calls 7 | tool calls 7 | healthy True | writ

## 7. Where it shows up in agents: v0 vs create_agent, same incident

v0 replays its recorded baseline run (sample 0) from its own cache: no API call.

In [11]:
v0 = agent_v0.run(inc, mode="replay", sample=0)
seq_v0 = [t.get("tool") for t in v0["trace"]]
seq_ca = [t.get("tool") for t in r["trace"]]
print("v0            :", seq_v0, "grade", agent_v0.grade(v0)["pass"])
print("create_agent  :", seq_ca, "grade", agent_v0.grade(r)["pass"])
print("same tool sequence:", seq_v0 == seq_ca)

v0            : ['check_health', 'read_logs', 'read_logs', 'rotate_secret', 'check_health', 'finish'] grade True
create_agent  : ['check_health', 'recent_deploys', 'read_logs', 'read_logs', 'rotate_secret', 'check_health', 'finish'] grade True
same tool sequence: False


Both pass; create_agent made one extra call, `recent_deploys`, second in line. That is one run each at temperature 1,
too small to rank them. Lesson 4.5 tests it properly.

**Identical:** model and settings, prompt, tools, limit, both nudges, the grader.
**Added:** a graph you can draw, a tool node that runs your tools, named slots for new behaviour.
**The price:** hook order and tool threads now belong to the framework, and both bit us (traps 2 and 3).

## 8. Traps
The four real problems of the rebuild, each as a cell that runs.

**Trap 1: the cache must key on everything that changes the reply.** LangChain's own LLM cache did not replay our runs,
so the cache became a `wrap_model_call` middleware keyed on the prompt, the conversation, the sample and the temperature.
Change any of them and it is a different key, so a new API call:

In [12]:
import hashlib, json
def key(system, convo, sample, temperature):
    k = json.dumps([system, convo, sample, temperature], sort_keys=True)
    return hashlib.sha256(k.encode()).hexdigest()[:24]

base = key("You are Nightwatch.", [["human", "PAGER: checkout error rate above 5%"]], 0, 1.0)
print("same inputs   :", base == key("You are Nightwatch.", [["human", "PAGER: checkout error rate above 5%"]], 0, 1.0))
print("sample 1      :", base == key("You are Nightwatch.", [["human", "PAGER: checkout error rate above 5%"]], 1, 1.0))
print("temperature 0 :", base == key("You are Nightwatch.", [["human", "PAGER: checkout error rate above 5%"]], 0, 0.0))

same inputs   : True
sample 1      : False
temperature 0 : False


**Trap 2: the order of the list.** Wrap hooks nest, first listed = outermost. Two wrap hooks that print on the way in and out:

In [13]:
order = []
def tracer(tag):
    @wrap_model_call(name=f"wrap_{tag}")
    def w(request, handler):
        order.append(f"{tag} in")
        out = handler(request)
        order.append(f"{tag} out")
        return out
    return w

mini_agent(fake(call("finish", 1, cause="x", root_cause_service="y")), [tracer("retry"), tracer("cache")]).invoke(
    {"messages": [HumanMessage("PAGER")]})
print(order)

['retry in', 'cache in', 'cache out', 'retry out']


**Trap 3: SQLite and the thread pool.** The tool node runs plain functions in a thread pool, but v0 opened its database
in the main thread. `nw_agent.run` reopens it with `check_same_thread=False`. The error, reproduced:

In [14]:
import sqlite3
from concurrent.futures import ThreadPoolExecutor

db = sqlite3.connect(":memory:")                        # created in the main thread
try:
    ThreadPoolExecutor(1).submit(lambda: db.execute("select 1").fetchall()).result()
except sqlite3.ProgrammingError as e:
    print("ProgrammingError:", str(e).split(". The object")[0])

db_ok = sqlite3.connect(":memory:", check_same_thread=False)
print("check_same_thread=False:", ThreadPoolExecutor(1).submit(lambda: db_ok.execute("select 1").fetchall()).result())

ProgrammingError: SQLite objects created in a thread can only be used in that same thread
check_same_thread=False: [(1,)]


**Trap 4: `functions.finish`.** Now and then the model calls a tool named `functions.finish`. No such tool exists, so the
tool node replies with an error naming the real tools. `return_direct` belongs to the real `finish` only, so the run goes
on, and each slip costs one of the 14 calls.

In [15]:
WROTE.clear()
out = mini_agent(fake(call("functions.finish", 1, cause="x", root_cause_service="y"),
                      call("finish", 2, cause="x", root_cause_service="y")), []).invoke({"messages": [HumanMessage("PAGER")]})
for m in out["messages"]:
    print(f"{m.type:6}", (m.tool_calls[0]["name"] if getattr(m, "tool_calls", None) else m.content)[:110])

human  PAGER
ai     functions.finish
tool   Error: functions.finish is not a valid tool, try one of [check_health, rotate_secret, finish].
ai     finish
tool   ok


## 9. Exercises
Every check cell runs on the fake model: no API.

### Exercise 1 (the video's exercise): swap the order

Make `swapped`: the list `[nudge_to_tools, limit, log_tool]` with `nudge_to_tools` and the limit swapped. Draw the graph
with `edges(mini_agent(fake(), swapped))`, then set `first_after` to the name of the after_model hook that now runs first
(`"nudge_to_tools"` or `"ModelCallLimitMiddleware"`).

In [16]:
limit = ModelCallLimitMiddleware(run_limit=14, exit_behavior="end")
swapped = [nudge_to_tools, limit, log_tool]   # YOUR CODE HERE: swap the first two
first_after = ""                               # YOUR CODE HERE

In [17]:
try:
    assert [m.name for m in swapped][:2] == ["ModelCallLimitMiddleware", "nudge_to_tools"], "swap the first two items"
    after_model_edge = [e for e in edges(mini_agent(fake(), swapped)) if e.startswith("model ")][0]
    print(after_model_edge)
    assert first_after and first_after in after_model_edge, f"read the edge out of model: {after_model_edge}"
    print("✓ Exercise 1 passed")
except AssertionError as e:
    print("✗ Exercise 1 not yet:", e)

✗ Exercise 1 not yet: swap the first two items


### Exercise 2: count the model calls with a wrap hook

Write `count_calls`, a `@wrap_model_call` middleware that adds 1 to `calls["n"]` each time the real model is called,
and returns the handler's result.

In [18]:
calls = {"n": 0}

# YOUR CODE HERE: define count_calls
count_calls = None

In [19]:
try:
    assert count_calls is not None, "define count_calls with @wrap_model_call"
    calls["n"] = 0
    WROTE.clear(); box["trace"].clear()
    out = mini_agent(fake(call("check_health", 1), AIMessage("hmm"), call("finish", 2, cause="x", root_cause_service="y")),
                     [nudge_to_tools, count_calls]).invoke({"messages": [HumanMessage("PAGER")]})
    n_ai = sum(isinstance(m, AIMessage) for m in out["messages"])
    assert calls["n"] == n_ai == 3, f"counted {calls['n']}, AI messages {n_ai}, expected 3"
    print("model calls counted:", calls["n"])
    print("✓ Exercise 2 passed")
except AssertionError as e:
    print("✗ Exercise 2 not yet:", e)

✗ Exercise 2 not yet: define count_calls with @wrap_model_call


### Exercise 3: block writes with a tool hook

Write `block_writes`, a `@wrap_tool_call` middleware: if the tool name is in `WRITES`, do **not** call the handler; return
`ToolMessage(content="BLOCKED: <name> needs a human", tool_call_id=request.tool_call["id"])`. Otherwise run the tool.

In [20]:
WRITES = {"rotate_secret", "restart_service", "rollback_deploy"}

# YOUR CODE HERE: define block_writes
block_writes = None

In [21]:
try:
    assert block_writes is not None, "define block_writes with @wrap_tool_call"
    WROTE.clear()
    out = mini_agent(fake(call("rotate_secret", 1, service="payments"), call("check_health", 2),
                          call("finish", 3, cause="expired_secret", root_cause_service="payments")),
                     [block_writes]).invoke({"messages": [HumanMessage("PAGER")]})
    tool_msgs = [m.content for m in out["messages"] if isinstance(m, ToolMessage)]
    print(tool_msgs)
    assert WROTE == [], f"the write ran: {WROTE}"
    assert tool_msgs[0].startswith("BLOCKED: rotate_secret"), f"first tool reply: {tool_msgs[0]}"
    assert tool_msgs[1].startswith("unhealthy"), "reads must still run"
    print("✓ Exercise 3 passed")
except AssertionError as e:
    print("✗ Exercise 3 not yet:", e)

✗ Exercise 3 not yet: define block_writes with @wrap_tool_call


### Exercise 4: where do the two runs part?

Write `first_diff(a, b)` that returns `(i, a[i], b[i])` for the first position where two tool sequences differ
(use `None` for a missing item), or `None` if they are equal. Try it on `seq_v0` and `seq_ca` from section 7.

In [22]:
def first_diff(a, b):
    # YOUR CODE HERE
    return None

In [23]:
try:
    v0_seq = ['check_health', 'read_logs', 'read_logs', 'rotate_secret', 'check_health', 'finish']
    ca_seq = ['check_health', 'recent_deploys', 'read_logs', 'read_logs', 'rotate_secret', 'check_health', 'finish']
    assert first_diff(v0_seq, ca_seq) == (1, "read_logs", "recent_deploys"), f"got {first_diff(v0_seq, ca_seq)}"
    assert first_diff(v0_seq, v0_seq) is None, "equal sequences give None"
    assert first_diff(["a"], ["a", "b"]) == (1, None, "b"), "a missing item is None"
    print("first difference:", first_diff(v0_seq, ca_seq))
    print("✓ Exercise 4 passed")
except AssertionError as e:
    print("✗ Exercise 4 not yet:", e)

✗ Exercise 4 not yet: got None


## Solutions (try first!)

In [24]:
limit = ModelCallLimitMiddleware(run_limit=14, exit_behavior="end")
swapped = [limit, nudge_to_tools, log_tool]
first_after = "nudge_to_tools"     # model --> nudge_to_tools.after_model: reverse list order

In [25]:
calls = {"n": 0}

@wrap_model_call
def count_calls(request, handler):
    calls["n"] += 1
    return handler(request)

In [26]:
WRITES = {"rotate_secret", "restart_service", "rollback_deploy"}

@wrap_tool_call
def block_writes(request, handler):
    name = request.tool_call["name"]
    if name in WRITES:
        return ToolMessage(content=f"BLOCKED: {name} needs a human", tool_call_id=request.tool_call["id"])
    return handler(request)

In [27]:
def first_diff(a, b):
    for i in range(max(len(a), len(b))):
        x = a[i] if i < len(a) else None
        y = b[i] if i < len(b) else None
        if x != y:
            return (i, x, y)
    return None